## Importing libraries

In [ ]:
import marimo as mo
import subprocess #  lets Python run terminal commands (we need this to run yt-dlp)
import json # lets Python read JSON data (yt-dlp returns data in JSON format)
import os # lets Phyton interact with the file system — create folders, list files, check if files exist, get file paths. 
from groq import Groq

## Setting up Tiktok Url

In [ ]:
url = "https://www.tiktok.com/@zachsfoods/video/7595255804333149470" # TikTok video URL

## Fetching Video Metadata

In [ ]:
result = subprocess.run(  # run yt-dlp command
    ["yt-dlp", "--dump-json", "--no-download", url], # get metadata only, no download
    capture_output=True, text=True  # capture output as text
)

data = json.loads(result.stdout)  # parse JSON response into dictionary

## Display Metadata

In [ ]:
print("Title:", data.get("title", "N/A"))  # video title
print("Uploader:", data.get("uploader", "N/A"))  # creator name
print("Description:", data.get("description", "N/A")) # video caption
print("Duration:", data.get("duration", "N/A"), "seconds") # video length
print("View count:", data.get("view_count", "N/A"))  # total views

Title: Honey Garlic Chicken!  #easyrecipe #cooking #easydinner #asianfood #r...
Uploader: zachsfoods
Description: Honey Garlic Chicken!  #easyrecipe #cooking #easydinner #asianfood #ricebowl  For the meat 🍗: 1lb chicken thighs 2 tsp canola oil Salt, Pepper to taste 2 tbsp cornstarch (enough to coat) For the sauce: 1.5 tbsp low sodium soy sauce 3 tbsp honey 2 tbsp water 1/2 tbsp sugar 1/2 tsp red pepper flakes 1/2 tsp paprika 3 garlic cloves (minced) 2 tbsp green onion whites
Duration: 27 seconds
View count: 11200000


## Download Video and Check for Subtitles

In [ ]:
os.makedirs("outputs", exist_ok=True)  # create outputs folder if it doesn't exist

subprocess.run([
    "yt-dlp",  # run yt-dlp
    "--write-subs",  # download subtitles if available
    "--sub-langs", "all",  # get all subtitle languages
    "--write-auto-subs",  # also get auto-generated subtitles
    "-o", "outputs/%(id)s.%(ext)s",  # save to outputs folder with video ID as filename
    url  # the TikTok URL
])

video_id = url.split("/")[-1].split("?")[0]  # extract video ID from URL
video_file = None  # will store video file path
subtitle_file = None  # will store subtitle file path

for f in os.listdir("outputs"):  # loop through downloaded files
    if video_id in f and f.endswith(".mp4"):  # find the video file
        video_file = f"outputs/{f}"
    if video_id in f and (".vtt" in f or ".srt" in f):  # find subtitle file
        subtitle_file = f"outputs/{f}"

print("Video:", video_file)  # show video path
print("Subtitles:", subtitle_file)  # show subtitle path (None if no subs)

[TikTok] Extracting URL: https://www.tiktok.com/@zachsfoods/video/7595255804333149470
[TikTok] 7595255804333149470: Downloading webpage


[info] 7595255804333149470: Downloading subtitles: eng-US
[info] 7595255804333149470: Downloading 1 format(s): bytevc1_1080p_1569149-1
Deleting existing file outputs/7595255804333149470.eng-US.vtt
[info] Writing video subtitles to: outputs/7595255804333149470.eng-US.vtt
[download] Destination: outputs/7595255804333149470.eng-US.vtt
[download] 100% of    1.15KiB in 00:00:00 at 20.78KiB/s  
[download] outputs/7595255804333149470.mp4 has already been downloaded
[download] 100% of    5.20MiB
Video: outputs/7595255804333149470.mp4
Subtitles: outputs/7595255804333149470.eng-US.vtt


## Extract Speech Text from Subtitles

In [ ]:
speech_text = ""

if subtitle_file:  # subtitles found, use them
    with open(subtitle_file, "r") as sub_file:  # renamed from f to sub_file
        lines = sub_file.readlines()
    for line in lines:  # filter out timestamps and metadata
        line = line.strip()
        if line and not line.startswith("WEBVTT") and "-->" not in line and not line.isdigit():
            speech_text += line + " "
    print("Source: TikTok subtitles")
else:  # no subtitles, will use Whisper later
    print("No subtitles found — will use Whisper")

print("\nSpeech text:")
print(speech_text.strip())

Source: TikTok subtitles

Speech text:
I'm an amateur cook trying to show you that cooking is really not that hard. And this is honey, garlic chicken. Start your chicken, salt, pepper, a little bit of oil, cornstarch. Just mix until that's all nice and coated. Soy sauce. Soy sauce, lots of honey, red pepper flakes, little paprika, water, garlic, more garlic, sugar. With that together, heat some oil, add your chicken in. Cook until 1 65 and nice and golden brown. Then in the same pan, add your green onion whites. As your butter melts down, add in your sauce. Mixture should start to thicken up quite a bit. And that's definitely quicker than takeout, which is great because


## Load LLM (Qwen2.5)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

print("Model loaded!")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 519.90it/s]


Model loaded!


## Extract Structured Recipe

In [ ]:
context = f"""
Video Title: {data.get("title", "")}
Video Description: {data.get("description", "")}
Speech Transcript: {speech_text.strip()}
"""

prompt = f"""You are a recipe extraction AI. Analyse the following TikTok video data and:

1. Determine if this is a cooking video (YES/NO)
2. If YES, extract a structured recipe in JSON format with these fields:
   - dish_name
   - cuisine_type
   - difficulty (easy/medium/hard)
   - prep_time
   - cook_time
   - servings
   - ingredients (list with quantity and item)
   - steps (numbered list)
   - halal_status (halal/not_halal/uncertain with reason)

Video Data:
{context}

Respond ONLY in valid JSON."""

messages = [
    {"role": "system", "content": "You are a recipe extraction assistant. Always respond in valid JSON only."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=1024, temperature=0.1, do_sample=True)
response = tokenizer.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

print(response)

{
  "is_cooking_video": true,
  "recipe": {
    "dish_name": "Honey Garlic Chicken",
    "cuisine_type": "Asian",
    "difficulty": "Easy",
    "prep_time": "10 minutes",
    "cook_time": "15-20 minutes",
    "servings": "4 people",
    "ingredients": [
      {"quantity": "1 lb", "item": "chicken thighs"},
      {"quantity": "2 tsp", "item": "canola oil"},
      {"quantity": "2 tsp", "item": "cornstarch"},
      {"quantity": "1.5 tbsp", "item": "low sodium soy sauce"},
      {"quantity": "3 tbsp", "item": "honey"},
      {"quantity": "2 tbsp", "item": "water"},
      {"quantity": "1/2 tbsp", "item": "sugar"},
      {"quantity": "1/2 tsp", "item": "red pepper flakes"},
      {"quantity": "1/2 tsp", "item": "paprika"},
      {"quantity": "3 garlic cloves", "item": "minced"}
    ],
    "steps": [
      "Start by marinating the chicken thighs in salt, pepper, and cornstarch.",
      "Heat oil in a large skillet over medium-high heat.",
      "Add the chicken to the skillet and cook until i